In [ ]:
def f_exact_gs_energy(path,ex_n,ll,wT):
    e_out = fr"{path}/exec_{ex_n}/exact_{ll}_{wT}.csv"
    df_e = pd.read_csv(e_out)
    exact_gs_energy = df_e.iloc[0, 2];
    return float(exact_gs_energy)

In [ ]:
def f_jastrow(path,ex_n, ll, wT, it):
    j_out = fr"{path}/exec_{ex_n}/jastrow_ID{ex_n}_L_{ll}_W_{wT}_IT_{it}.log"
    with open(j_out) as f:
        data = json.load(f)
    iters  = data["Energy"]["iters"]
    energy = data["Energy"]["Mean"]
    if isinstance(energy, list):
        energy_Jastrow = [e["real"] for e in energy]
    else:
        energy_Jastrow = energy["real"]
    stride = 5
    iters_Jastrow_ds, energy_Jastrow_ds = downsample(iters, energy_Jastrow, stride) 
    return iters_Jastrow_ds, energy_Jastrow_ds

In [ ]:
def f_ffnn(path,ex_n, ll, wT, it):
    f_out = fr"{path}/exec_{ex_n}/ffnn_ID_{ex_n}_L_{ll}_W_{wT}_IT_{it}.log"
    with open(f_out) as f:
        data = json.load(f)
    iters  = data["Energy"]["iters"]
    energy = data["Energy"]["Mean"]
    if isinstance(energy, list):
        energy_Net = [e["real"] for e in energy]
    else:
        energy_Net = energy["real"]
    stride = 5
    iters_Net_ds, energy_Net_ds = downsample(iters, energy_Net, stride)  
    return iters_Net_ds, energy_Net_ds

In [ ]:
def w_net(path,ex_n, ll, wT, it, tNeur,tF ):
    f_out   = fr"{path}/exec_{ex_n}/ffnn_ID_{ex_n}_L_{str(ll)}_W_{str(wT)}_IT_{str(it)}_{tNeur}_{tF}.csv"
    df    = pd.read_csv(f_out)
    return df

In [ ]:
def jobs_egs(ll, wT,it, path, j_lim):  
    F_Egs = []
    for i in range (0,j_lim):
        ex_n = digts(i)
        try:
            iters_Net_ds,energy_Net_ds = f_ffnn(path,ex_n, ll, wT, it)  
            F_Egs.append(energy_Net_ds[-1])
        except:
            print(i)       
    df = pd.DataFrame({
        'id': range(1, len(F_Egs) + 1),   
        'F_Egs': F_Egs                    
    })

    egs = df["F_Egs"]
    median_val  = egs.median()
    closest_idx = (egs - median_val).abs().idxmin()
  
    id_median   = df.loc[closest_idx, "id"]
    egs_median  = df.loc[closest_idx, "F_Egs"]

    return df, float(egs_median), float(id_median)